# Parallel Iterative SLSQP — All Test Cases
Runs `iterative_parallel` on every test case generated by `generate_test_cases.ipynb`.
Results are saved under `OUTPUT_DIR` alongside the input `.npy` files.

## Imports

In [ ]:
import os
import time
import glob

import numpy as np
import matplotlib.pyplot as plt

from modules.dvfopt import jacobian_det2D, iterative_parallel
from modules.dvfviz import (
    plot_deformations,
    plot_grid_before_after,
    plot_checkerboard_before_after,
    plot_neg_jdet_neighborhoods,
)
from modules.testcases import (
    SYNTHETIC_CASES, RANDOM_DVF_CASES, REAL_DATA_SLICES,
    make_deformation, make_random_dvf, load_slice,
)

In [ ]:
INPUT_DIR = "data/test_cases"
OUTPUT_DIR = "output/test_cases/results/parallel_slsqp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Jacobian determinant threshold — pixels with Jdet <= threshold are corrected
JDET_THRESHOLD = 0.1 # 0.01

## Helper

In [ ]:
def run_and_show(key, deformation, msample=None, fsample=None, **kwargs):
    """Run parallel correction, save outputs, print summary, and plot."""
    save_dir = f"{OUTPUT_DIR}/{key}"

    H, W = deformation.shape[-2:]
    jac_init = jacobian_det2D(np.stack([deformation[-2, 0], deformation[-1, 0]]))
    n_neg_init = int((jac_init <= 0).sum())
    n_below_thresh_init = int((jac_init <= JDET_THRESHOLD).sum())
    min_jdet_init = float(jac_init.min())

    print(f"\n{'='*70}")
    print(f"  {key}  |  {H}x{W}  |  neg Jdet: {n_neg_init}  "
          f"|  below threshold ({JDET_THRESHOLD}): {n_below_thresh_init}  "
          f"|  min Jdet: {min_jdet_init:.4f}")
    print(f"{'='*70}")

    t0 = time.perf_counter()
    phi = iterative_parallel(
        deformation.copy(), save_path=save_dir, verbose=1,
        threshold=JDET_THRESHOLD, **kwargs
    )
    elapsed = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    n_below_thresh_final = int((jac_final <= JDET_THRESHOLD).sum())
    min_jdet_final = float(jac_final.min())
    l2 = float(np.sqrt(np.sum((phi - np.stack([deformation[-2, 0], deformation[-1, 0]]))**2)))

    print(f"  Time: {elapsed:.2f}s  |  neg Jdet: {n_neg_init} -> {n_neg_final}  "
          f"|  below threshold: {n_below_thresh_init} -> {n_below_thresh_final}  "
          f"|  min Jdet: {min_jdet_init:.4f} -> {min_jdet_final:.4f}  |  L2: {l2:.4f}")

    plot_deformations(msample, fsample, deformation, phi, title=key)
    plot_grid_before_after(deformation, phi, title=key)
    plot_grid_before_after(deformation, phi, title=key, inverse=True)
    plot_checkerboard_before_after(deformation, phi, title=key)
    plot_neg_jdet_neighborhoods(deformation, phi, title=key)
    return phi

## Case 1 — Small synthetic grids

In [ ]:
for key in ["01a_10x10_crossing", "01b_10x10_opposite", "01c_20x40_edges", "01d_20x40_crossing"]:
    deformation, ms, fs = make_deformation(key)
    run_and_show(key, deformation, ms, fs)

## Case 1e–f — Random DVFs

In [ ]:
# Skipped — random DVFs take too long
# for key in ["01e_20x20_random_spirals", "01f_20x20_random_seed_42"]:
#     dvf = make_random_dvf(key)
#     run_and_show(key, dvf)

## Case 2 — Real data slices (downscaled)

In [ ]:
for key, cfg in REAL_DATA_SLICES.items():
    if cfg["scale_factor"] < 1.0:  # only downscaled for speed
        deformation, ms, fs = load_slice(cfg["slice_idx"], scale_factor=cfg["scale_factor"])
        run_and_show(key, deformation, ms, fs)

## Case 3 — Additional synthetic grids

In [ ]:
for key in ["03a_10x10_opposite", "03b_10x10_crossing", "03c_20x20_opposite", "03d_20x20_crossing"]:
    deformation, ms, fs = make_deformation(key)
    run_and_show(key, deformation, ms, fs)

### Random DVF variants

In [ ]:
# Skipped — random DVFs take too long
# for key in ["03a_10x10_random_seed_42", "03c_20x20_random_seed_42"]:
#     dvf = make_random_dvf(key)
#     run_and_show(key, dvf)

## Summary — collect all results

In [ ]:
result_dirs = sorted(glob.glob(f"{OUTPUT_DIR}/*/results.txt"))

print(f"{'Case':<35s}  {'Grid':>9s}  {'Time (s)':>9s}  {'L2':>10s}  {'Neg init':>8s}  {'Neg final':>9s}  {'Min Jdet':>10s}")
print("-" * 100)

for path in result_dirs:
    case_name = os.path.basename(os.path.dirname(path))
    lines = open(path).read()

    def _extract(label):
        for line in lines.splitlines():
            if label in line:
                return line.split(":")[-1].strip().split()[0]
        return "?"

    grid = _extract("resolution")
    runtime = _extract("run-time")
    l2 = _extract("Final L2")
    neg_start = _extract("Starting number of non-positive")
    neg_final = _extract("Final number of non-positive")
    min_jdet = _extract("Final Jacobian determinant minimum")

    print(f"{case_name:<35s}  {grid:>9s}  {runtime:>9s}  {l2:>10s}  {neg_start:>8s}  {neg_final:>9s}  {min_jdet:>10s}")